In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Forecast Errors and the Value of Foresight

A battery earns money by buying electricity when it is cheap and selling it when it is
expensive. How much it earns depends on how well it knows tomorrow's prices.

Two things limit that knowledge, and they are easy to confuse:

- **Limited foresight**: the operator only sees a few hours ahead.
- **Forecast error**: what the operator sees is wrong.

This example separates them. We model a 20 MW / 80 MWh battery trading on a day-ahead
market as a price taker. First we compute the profit it would earn knowing every price in
advance. That is the benchmark, often called the intrinsic value of the asset. Then we
operate the battery day by day against forecasts that are refreshed every morning and that
get the prices wrong. The gap between the two numbers is the value that the forecast
error destroys.

The [rolling horizon example](rolling-horizon.ipynb) covers limited foresight with
perfect data. Here we add the forecast error.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pypsa

rng = np.random.default_rng(42)

## A synthetic day-ahead price

We use two weeks of hourly prices built from three parts: a daily shape with a midday
solar dip and an evening peak, a level that drifts from day to day, and a small hourly
noise.

In [ ]:
days = 14
snapshots = pd.date_range("2030-06-01", periods=days * 24, freq="h")

daily_shape = np.array(
    [
        [-15, -22, -27, -30, -30, -24],
        [-10, 12, 22, 5, -20, -40],
        [-55, -58, -50, -30, 0, 25],
        [45, 55, 45, 25, 5, -8],
    ],
    dtype=float,
).ravel()
daily_shape -= daily_shape.mean()
level = 65 + rng.normal(0, 12, days).cumsum() * 0.4

price = pd.Series(
    level.repeat(24) + np.tile(daily_shape, days) + rng.normal(0, 4, len(snapshots)),
    index=snapshots,
    name="price",
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3), layout="constrained")

price.plot(ax=ax, color="black", linewidth=1)
ax.axhline(0, color="grey", linewidth=0.5)
ax.set_ylabel("EUR/MWh")
ax.set_xlabel("");

## A price-taker battery

The market is a single bus. The battery is a `StorageUnit` attached to it. The market
itself is one `Generator` whose output is allowed to go negative, which we allow with
`p_min_pu=-1`:

- positive output means the market delivers power to us, so we **buy**,
- negative output means the market absorbs power from us, so we **sell**.

Its `marginal_cost` is the spot price. Buying therefore costs the spot price and selling
earns it. Because the price is fixed data rather than a result of the optimisation, the
battery is a **price taker**. Profit is the negative
of the objective value.

The battery is not cyclic, which lets us hand its state of charge from one day to the
next. It starts empty.

In [ ]:
def build_network(prices):
    n = pypsa.Network(snapshots=prices.index)

    n.add("Carrier", ["electricity", "spot market", "battery"])
    n.add("Bus", "market", carrier="electricity")

    n.add(
        "Generator",
        "spot market",
        bus="market",
        carrier="spot market",
        p_nom=20,
        p_min_pu=-1,
        marginal_cost=prices,
    )

    n.add(
        "StorageUnit",
        "battery",
        bus="market",
        carrier="battery",
        p_nom=20,
        max_hours=4,
        efficiency_store=0.95,
        efficiency_dispatch=0.95,
    )

    return n

Settling a schedule means valuing every traded megawatt hour at the price that actually
occurred. Buying is a positive market output and costs money, so the profit carries a
minus sign.

In [ ]:
def settle(dispatch, prices):
    """Profit in EUR of a market schedule valued at the given prices."""
    return -(dispatch * prices[dispatch.index]).sum()

## Benchmark: perfect foresight

We solve the whole fortnight in one go with the prices that actually occur. No trader can
beat this, so it is the ceiling against which everything else is measured.

In [ ]:
n = build_network(price)
n.optimize()

profit_perfect = settle(n.generators_t.p["spot market"], price)
profit_perfect

The battery charges in the cheap hours and discharges into the peaks almost every day.
Since the marginal price of the bus equals the spot price here,
[`n.statistics.revenue()`][pypsa.statistics.StatisticsAccessor.revenue] reports the same
trading result with the opposite sign.

In [ ]:
revenue = n.statistics.revenue(components="Generator")

pd.Series(
    {
        "full cycles": n.storage_units_t.p_dispatch["battery"].sum() / 80,
        "profit (EUR)": profit_perfect,
        "statistics revenue (EUR)": revenue.iloc[0],
    }
).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3), layout="constrained")

n.storage_units_t.state_of_charge.plot(ax=ax, color="black", linewidth=1)
ax.axhline(0, color="grey", linewidth=0.5)
ax.set_ylabel("MWh")
ax.set_xlabel("");

## Forecasts that are refreshed every morning

Every morning the trader receives a new price forecast for the next two days. The forecast
is the realised price plus an error that grows with lead time: the next hour is almost
known, the day after tomorrow much less so. We scale the standard deviation of the error
with the square root of the lead time, so that one day ahead it equals `sigma`.

The hourly errors are drawn independently. This distorts the *shape* of the daily price
profile, which is what a battery trades on. For contrast we also build an error that
drifts as a random walk and therefore mostly shifts the price *level*. The last section
shows why that distinction decides how much money is lost.

In [ ]:
def make_forecast(prices, issue, horizon, sigma, rng, kind="shape"):
    """Forecast of the next `horizon` hours, as seen at time `issue`."""
    target = prices.loc[issue:].iloc[:horizon]
    lead = np.arange(len(target)) + 1

    if kind == "shape":
        error = rng.standard_normal(len(target)) * np.sqrt(lead / 24)
    else:
        error = rng.standard_normal(len(target)).cumsum() / np.sqrt(24)

    return target + sigma * error

In [ ]:
issue = snapshots[72]
example = make_forecast(price, issue, 48, sigma=20, rng=np.random.default_rng(1))

fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")

price[example.index].plot(ax=ax, color="black", linewidth=1.5)
example.plot(ax=ax, color="tab:orange", linewidth=1.5)
ax.annotate("realised", (0.02, 0.92), xycoords="axes fraction", color="black")
ax.annotate("forecast", (0.02, 0.82), xycoords="axes fraction", color="tab:orange")
ax.set_ylabel("EUR/MWh")
ax.set_xlabel("");

## Trading day by day

Every morning we repeat four steps:

1. write the new forecast into the `marginal_cost` of the market generator,
2. set the state of charge to where yesterday's trading left the battery,
3. optimise the next 48 hours,
4. keep only the first 24 hours and discard the rest, which tomorrow's forecast will
   decide again.

The loop is written out rather than handed to
[`n.optimize.optimize_with_rolling_horizon()`][pypsa.optimization.OptimizationAccessor.optimize_with_rolling_horizon]
because that helper re-solves the same data in every window. Here the data itself has to
change between the solves, since each window sees a different forecast.

In [ ]:
def trade(prices, sigma, horizon=48, commit=24, kind="shape", seed=0):
    """Operate the battery against forecasts reissued every `commit` hours."""
    n = build_network(prices)
    rng = np.random.default_rng(seed)

    soc = 0.0
    committed = []
    expected = 0.0

    for issue in prices.index[::commit]:
        window = prices.index[prices.index >= issue][:horizon]
        forecast = make_forecast(prices, issue, horizon, sigma, rng, kind)

        n.generators_t.marginal_cost.loc[window, "spot market"] = forecast
        n.storage_units.loc["battery", "state_of_charge_initial"] = soc

        n.optimize(window)

        today = window[:commit]
        committed.append(
            pd.DataFrame(
                {
                    "market": n.generators_t.p.loc[today, "spot market"],
                    "soc": n.storage_units_t.state_of_charge.loc[today, "battery"],
                }
            )
        )
        expected += settle(committed[-1].market, forecast)
        soc = committed[-1].soc.iloc[-1]

    return pd.concat(committed), expected

## What the forecast error costs

We run the loop with an error of 20 EUR/MWh one day ahead, which is a realistic order of
magnitude for day-ahead price forecasts. Two profits come out of it: the one the optimiser
expected while it was deciding, and the one the market actually paid.

In [ ]:
result, expected = trade(price, sigma=20)

pd.Series(
    {
        "Perfect foresight": profit_perfect,
        "Expected under forecast": expected,
        "Realised": settle(result.market, price),
    }
).round(0)

The realised profit falls short of perfect foresight. The expected profit, in contrast,
lies *above* it: wherever the forecast error invents
a price spread, the optimiser trades on it and books the imagined margin. Noise always
looks like opportunity to an optimiser that trusts its input.

Looking at three days of operation shows where the money is lost. The forecast-driven
battery charges and discharges at roughly the right times, but it misjudges which peak is
worth waiting for and how deep the midday trough will be.

In [ ]:
window = snapshots[48:120]

fig, axes = plt.subplots(
    2, 1, figsize=(9, 4), sharex=True, height_ratios=[1, 1.4], layout="constrained"
)

price[window].plot(ax=axes[0], color="black", linewidth=1)
axes[0].set_ylabel("EUR/MWh")

n.storage_units_t.state_of_charge.loc[window, "battery"].plot(
    ax=axes[1], color="black", linewidth=1.5
)
result.soc[window].plot(ax=axes[1], color="tab:orange", linewidth=1.5)
axes[1].set_ylim(0, 110)
axes[1].annotate(
    "perfect foresight", (0.02, 0.93), xycoords="axes fraction", color="black"
)
axes[1].annotate(
    "forecast driven", (0.25, 0.93), xycoords="axes fraction", color="tab:orange"
)
axes[1].set_ylabel("state of charge (MWh)")
axes[1].set_xlabel("");

## Is the horizon or the forecast the binding constraint?

Before blaming the forecast, we check how much the limited horizon costs on its own. We
rerun the loop with a perfect forecast (`sigma=0`) and shorten the window.

In [ ]:
horizons = {}
for horizon, commit in [(12, 12), (24, 24)]:
    schedule, _ = trade(price, sigma=0, horizon=horizon, commit=commit)
    horizons[f"{horizon} h"] = settle(schedule.market, price) / profit_perfect

In [ ]:
pd.Series(horizons).round(3)

One day of foresight is already enough for a four-hour battery: with a perfect forecast it
captures practically all of the perfect-foresight profit. The twelve-hour window loses more,
because it cuts the day in two and cannot move energy across the cut. Long-duration storage behaves very
differently, as the [water value example](water-value.ipynb) shows, where foresight
shorter than a season costs a lot and has to be repaired with a value on stored energy.

So in this case the horizon is nearly free and the forecast is what costs money.

## How much is a better forecast worth?

We sweep the forecast error and record both profits, once for the error that distorts the
shape of the daily profile and once for the error that only drifts the level.

In [ ]:
sigmas = [10, 20, 40]
records = {}
for kind in ["shape", "level"]:
    for sigma in sigmas:
        schedule, expected = trade(price, sigma=sigma, kind=kind)
        records[kind, sigma] = {
            "realised": settle(schedule.market, price) / profit_perfect,
            "expected": expected / profit_perfect,
        }

value = pd.DataFrame(records).T
value.index.names = ["error", "sigma"]

In [ ]:
value.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")

styles = {"shape": ("tab:orange", 12), "level": ("tab:blue", -18)}
for kind, (color, offset) in styles.items():
    rows = value.loc[kind]
    ax.plot(rows.index, rows.realised, color=color, marker="o")
    ax.plot(
        rows.index, rows.expected, color=color, linestyle="--", marker="o", alpha=0.6
    )
    ax.annotate(
        f"{kind} error",
        (rows.index[-1], rows.realised.iloc[-1]),
        xytext=(-8, offset),
        textcoords="offset points",
        color=color,
        ha="right",
    )

ax.axhline(1, color="black", linewidth=0.8)
ax.annotate(
    "perfect foresight", (sigmas[0], 1), xytext=(5, 6), textcoords="offset points"
)
ax.set_xlabel("forecast error one day ahead (EUR/MWh)")
ax.set_ylabel("share of perfect-foresight profit")
ax.set_title("solid: realised profit, dashed: profit the optimiser expected");